# 03 — Two-Stage Default · REG · enet (Stage 2 conditional)

Stage 2 회귀 (y>0 only conditional, `E[Y|Y>0,x]`) ElasticNet — PP + X scaling + y target_transform + HP joint Optuna.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/03_two_stage/default/reg/enet/`
- **PP**: PP joint Optuna (strategy_common §3 — ENet은 PP_FIXED 안 씀, 범위 탐색)
- **X scaling**: 5종 categorical (`StandardScaler / RobustScaler / YeoJohnson / Quantile / Hybrid`)
- **y target_transform**: 4종 categorical (`none / log1p / yeo-johnson / quantile`) — fold별 fit
- **Y_POSITIVE_ONLY**: True (y>0 die만 학습)
- **HPO**: N_TRIALS=200 (joint이라 비용 큼), anchor enqueue + Wide search
- **Sampler/Pruner/Timeout**: §4·§25


## 1. 환경 설정 + import

In [ ]:
import os, sys, io, contextlib

GDRIVE_CODE_ID         = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'
GDRIVE_DATASET_ID      = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'
GDRIVE_MODELING_ID     = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'  # ★ Colab 사용 시 신규 modeling.zip ID 입력

try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/hpo.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

PP_DIR = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_DIR not in sys.path:
    sys.path.insert(0, PP_DIR)
MOD_DIR = os.path.join(PROJECT_ROOT, '3_modeling')
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

from modules import preprocess, hpo

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer, QuantileTransformer
from sklearn.model_selection import KFold

# HybridScaler — 2_preprocessing/scaling.py
from scaling import HybridScaler

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')


## 2. 실험 설정

In [ ]:
REG_MODEL_NAME = 'enet'
EXP_ID   = f'ts-reg-{REG_MODEL_NAME}-002'
EXP_MEMO = 'Two-Stage default · REG · enet · y>0 conditional · PP+scaling+y_transform joint'
USER     = 'jh'

N_TRIALS         = 200   # joint이라 비용 큼
N_FOLDS          = 5
N_STARTUP_TRIALS = 10
N_JOBS           = 14    # ★ ENet 단독 실행 (strategy_common §8)
TIMEOUT_SEC      = None  # ★ §25

CLIP_Y_EXTREME = True
Y_POSITIVE_ONLY = True

OUT_DIR = os.path.join(OUTPUT_DIR, '03_two_stage', 'default', 'reg', REG_MODEL_NAME)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# anchor (1차 reg-enet best, log1p ON 컨텍스트)
ENET_ANCHOR = {
    'missing_threshold':          0.5,
    'corr_threshold':             0.90,
    'add_indicator':              True,
    'indicator_threshold':        0.10,
    'spatial_max_dist':           5.0,
    'post_impute_corr_threshold': 0.97,
    'scaling':                    'RobustScaler',
    'target_transform':           'log1p',
    'alpha':                      0.000145,
    'l1_ratio':                   0.725,
    'max_iter':                   15000,
}

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS} | N_JOBS={N_JOBS} | TIMEOUT_SEC={TIMEOUT_SEC}')
print(f'Y_POSITIVE_ONLY={Y_POSITIVE_ONLY}')
print(f'OUT_DIR={OUT_DIR}')


## 3. 데이터 로드 + Y clip

In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

print('[target transform] Optuna가 trial별 자동 탐색 (none/log1p/yeo-johnson/quantile)')


## 4. KFold split + helper (scaling, target transform, PP)

- KFold는 unit-level (모든 trial 공유)
- `make_target_transformer`: fold별 train fold y에 fit (leakage 방지)
- ENet은 inverse 후 음수 가능 → `np.clip(0, None)`

In [ ]:
unit_ids_train = ys_input['train'][KEY_COL].unique()
_kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = []
for tr_idx, vl_idx in _kf.split(unit_ids_train):
    FOLDS.append((unit_ids_train[tr_idx], unit_ids_train[vl_idx]))
print(f'[KFold] {N_FOLDS} folds, unit 단위 분할')


def make_scaler(name):
    if name == 'StandardScaler':
        return StandardScaler()
    if name == 'RobustScaler':
        return RobustScaler()
    if name == 'YeoJohnson':
        return PowerTransformer(method='yeo-johnson', standardize=True)
    if name == 'Quantile':
        return QuantileTransformer(output_distribution='normal', random_state=SEED)
    if name == 'Hybrid':
        return HybridScaler(skew_threshold=10.0)
    raise ValueError(f'Unknown scaling: {name!r}')


def make_target_transformer(name, y_train_arr):
    """fold별 train fold y에 fit한 (forward_fn, inverse_fn) 반환.

    선형 모델은 inverse 후 음수 가능 → 모든 inverse는 np.clip(0, None) 적용.
    """
    if name == 'none':
        return (
            lambda y: np.asarray(y, dtype=float),
            lambda y: np.clip(np.asarray(y, dtype=float), 0.0, None),
        )
    if name == 'log1p':
        return (
            lambda y: np.log1p(np.asarray(y, dtype=float)),
            lambda y: np.clip(np.expm1(np.asarray(y, dtype=float)), 0.0, None),
        )
    if name == 'yeo-johnson':
        pt = PowerTransformer(method='yeo-johnson', standardize=False)
        pt.fit(np.asarray(y_train_arr, dtype=float).reshape(-1, 1))
        return (
            lambda y: pt.transform(np.asarray(y, dtype=float).reshape(-1, 1)).ravel(),
            lambda y: np.clip(pt.inverse_transform(np.asarray(y, dtype=float).reshape(-1, 1)).ravel(), 0.0, None),
        )
    if name == 'quantile':
        n_q = min(1000, len(y_train_arr))
        qt = QuantileTransformer(output_distribution='normal', n_quantiles=n_q, random_state=SEED)
        qt.fit(np.asarray(y_train_arr, dtype=float).reshape(-1, 1))
        return (
            lambda y: qt.transform(np.asarray(y, dtype=float).reshape(-1, 1)).ravel(),
            lambda y: np.clip(qt.inverse_transform(np.asarray(y, dtype=float).reshape(-1, 1)).ravel(), 0.0, None),
        )
    raise ValueError(f'Unknown target_transform: {name!r}')


def run_pp_silent(pp_params):
    """preprocess.run 호출. corr_keep_by/post_impute_corr_keep_by는 'std' 고정 (leakage 방지)."""
    pp_full = dict(pp_params, corr_keep_by='std', post_impute_corr_keep_by='std')
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        out = preprocess.run(xs.copy(), ys_input, feat_cols, xs_dict, params=pp_full)
    return out


print('[helper] make_scaler / make_target_transformer / run_pp_silent 정의 완료')


## 5. Optuna joint HPO (PP + scaling + target_transform + enet HP)

- objective: y>0 die만 fit, OOF unit RMSE
- anchor enqueue (1차 best HP 시작점)

In [ ]:
y_train_unit_df = ys_input['train']
y_train_unit    = y_train_unit_df.set_index(KEY_COL)[TARGET_COL]


def _broadcast_y_to_die(xs_split, y_unit_series):
    return xs_split[KEY_COL].map(y_unit_series).values.astype(float)


def _aggregate_die_to_unit_mean(xs_split, die_pred):
    df = pd.DataFrame({KEY_COL: xs_split[KEY_COL].values, 'pred': die_pred})
    return df.groupby(KEY_COL, sort=False)['pred'].mean().reset_index()


def objective(trial):
    # ── PP 6축 ──
    pp_params = {
        'missing_threshold':          trial.suggest_float('missing_threshold', 0.30, 0.90),
        'corr_threshold':             trial.suggest_float('corr_threshold', 0.88, 0.98),
        'add_indicator':              trial.suggest_categorical('add_indicator', [True, False]),
        'indicator_threshold':        trial.suggest_float('indicator_threshold', 0.05, 0.20),
        'spatial_max_dist':           trial.suggest_float('spatial_max_dist', 1.0, 6.0),
        'post_impute_corr_threshold': trial.suggest_float('post_impute_corr_threshold', 0.96, 0.99),
    }
    # ── X scaling 5종 ──
    scaling_name = trial.suggest_categorical(
        'scaling', ['StandardScaler', 'RobustScaler', 'YeoJohnson', 'Quantile', 'Hybrid']
    )
    # ── y target_transform 4종 (strategy_common §3) ──
    target_transform_name = trial.suggest_categorical(
        'target_transform', ['none', 'log1p', 'yeo-johnson', 'quantile']
    )
    # ── enet HP ──
    enet_hp = dict(
        alpha=trial.suggest_float('alpha', 1e-7, 1e-3, log=True),
        l1_ratio=trial.suggest_float('l1_ratio', 0.50, 0.95),
        max_iter=trial.suggest_int('max_iter', 8000, 20000, step=1000),
        tol=1e-6, selection='random', precompute=True, random_state=SEED,
    )

    # ── PP ──
    try:
        out = run_pp_silent(pp_params)
    except Exception as e:
        raise optuna.exceptions.TrialPruned(f'PP failed: {e}')
    xs_train_c      = out['xs_train']
    feat_cols_clean = out['feat_cols']

    y_die_orig = _broadcast_y_to_die(xs_train_c, y_train_unit)

    n_tr = len(xs_train_c)
    oof = np.full(n_tr, np.nan)

    # ── fold ──
    for tr_units, vl_units in FOLDS:
        tr_mask = xs_train_c[KEY_COL].isin(set(tr_units)).values
        vl_mask = xs_train_c[KEY_COL].isin(set(vl_units)).values

        X_tr = xs_train_c.loc[tr_mask, feat_cols_clean].values
        X_vl = xs_train_c.loc[vl_mask, feat_cols_clean].values

        # ── Y_POSITIVE_ONLY: y>0 die만 fit (val은 전체 die에 예측) ──
        y_tr_orig = y_die_orig[tr_mask]
        if Y_POSITIVE_ONLY:
            pos_mask = y_tr_orig > 0
            X_tr_fit = X_tr[pos_mask]
            y_tr_fit = y_tr_orig[pos_mask]
        else:
            X_tr_fit = X_tr
            y_tr_fit = y_tr_orig

        # target transformer는 train fold y(>0)에만 fit (leakage 방지)
        forward_fn, inverse_fn = make_target_transformer(target_transform_name, y_tr_fit)
        y_fit = forward_fn(y_tr_fit)

        scaler = make_scaler(scaling_name)
        X_tr_s = scaler.fit_transform(X_tr_fit)
        X_vl_s = scaler.transform(X_vl)

        try:
            model = ElasticNet(**enet_hp)
            model.fit(X_tr_s, y_fit)
            pred_t = model.predict(X_vl_s)
        except Exception as e:
            raise optuna.exceptions.TrialPruned(f'enet fit/predict failed: {e}')
        oof[vl_mask] = inverse_fn(pred_t)

    if np.isnan(oof).any():
        raise RuntimeError('OOF has NaN — fold coverage bug')

    unit_pred = _aggregate_die_to_unit_mean(xs_train_c, oof)
    aligned   = unit_pred.set_index(KEY_COL)['pred'].loc[y_train_unit.index]
    train_rmse = float(np.sqrt(np.mean((aligned.values - y_train_unit.values) ** 2)))
    trial.set_user_attr('train_rmse', train_rmse)
    trial.set_user_attr('n_features_after_pp', len(feat_cols_clean))
    return train_rmse


# study + anchor enqueue
study = optuna.create_study(
    direction='minimize',
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    load_if_exists=False,
    sampler=TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS),
    pruner=MedianPruner(n_warmup_steps=10),
)
study.set_user_attr('exp_id', EXP_ID)
study.set_user_attr('exp_memo', EXP_MEMO)
study.set_user_attr('user', USER)
study.set_user_attr('y_positive_only', Y_POSITIVE_ONLY)
study.set_user_attr('anchor', ENET_ANCHOR)
study.enqueue_trial(ENET_ANCHOR)

study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, show_progress_bar=True)

best_params = dict(study.best_trial.params)
best_value  = float(study.best_value)
print(f'\n[HPO 완료] best OOF RMSE = {best_value:.6f}')
print(f'best_params = {best_params}')


## 6. Best trial refit (5-fold + die-level pred 캐쳐)

In [ ]:
PP_KEYS = ('missing_threshold', 'corr_threshold', 'add_indicator',
           'indicator_threshold', 'spatial_max_dist', 'post_impute_corr_threshold')
best_pp               = {k: best_params[k] for k in PP_KEYS if k in best_params}
best_scaling          = best_params.get('scaling', 'RobustScaler')
best_target_transform = best_params.get('target_transform', 'log1p')
best_enet             = {k: best_params[k] for k in ('alpha', 'l1_ratio', 'max_iter')}
best_enet.update(tol=1e-6, selection='random', precompute=True, random_state=SEED)

pp_full = dict(best_pp, corr_keep_by='std', post_impute_corr_keep_by='std')
out = preprocess.run(xs.copy(), ys_input, feat_cols, xs_dict, params=pp_full)
xs_train_c      = out['xs_train']
xs_val_c        = out['xs_val']
xs_test_c       = out['xs_test']
feat_cols_clean = out['feat_cols']

y_die_orig = _broadcast_y_to_die(xs_train_c, y_train_unit)

n_tr, n_vl, n_te = len(xs_train_c), len(xs_val_c), len(xs_test_c)
oof_pred  = np.full(n_tr, np.nan)
val_pred  = np.zeros(n_vl)
test_pred = np.zeros(n_te)

fold_models  = []
fold_scalers = []

for i, (tr_units, vl_units) in enumerate(FOLDS):
    tr_mask = xs_train_c[KEY_COL].isin(set(tr_units)).values
    vl_mask = xs_train_c[KEY_COL].isin(set(vl_units)).values

    X_tr = xs_train_c.loc[tr_mask, feat_cols_clean].values
    X_vl = xs_train_c.loc[vl_mask, feat_cols_clean].values
    X_v  = xs_val_c[feat_cols_clean].values
    X_te = xs_test_c[feat_cols_clean].values

    y_tr_orig = y_die_orig[tr_mask]
    if Y_POSITIVE_ONLY:
        pos_mask = y_tr_orig > 0
        X_tr_fit = X_tr[pos_mask]
        y_tr_fit = y_tr_orig[pos_mask]
    else:
        X_tr_fit = X_tr
        y_tr_fit = y_tr_orig

    forward_fn, inverse_fn = make_target_transformer(best_target_transform, y_tr_fit)
    y_fit = forward_fn(y_tr_fit)

    scaler = make_scaler(best_scaling)
    X_tr_s = scaler.fit_transform(X_tr_fit)
    X_vl_s = scaler.transform(X_vl)
    X_v_s  = scaler.transform(X_v)
    X_te_s = scaler.transform(X_te)

    model = ElasticNet(**best_enet)
    model.fit(X_tr_s, y_fit)

    oof_pred[vl_mask] = inverse_fn(model.predict(X_vl_s))
    val_pred  += inverse_fn(model.predict(X_v_s))  / N_FOLDS
    test_pred += inverse_fn(model.predict(X_te_s)) / N_FOLDS

    fold_models.append(model)
    fold_scalers.append(scaler)
    print(f'[refit fold {i+1}/{N_FOLDS}] tr_units={len(tr_units)}, vl_units={len(vl_units)}, fit_n={len(X_tr_fit):,}')

oof_unit  = _aggregate_die_to_unit_mean(xs_train_c, oof_pred)
val_unit  = _aggregate_die_to_unit_mean(xs_val_c,   val_pred)
test_unit = _aggregate_die_to_unit_mean(xs_test_c,  test_pred)

y_val_true  = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_true = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

oof_rmse  = float(np.sqrt(np.mean((oof_unit.set_index(KEY_COL)['pred'].loc[y_train_unit.index].values - y_train_unit.values)**2)))
val_rmse  = float(np.sqrt(np.mean((val_unit.set_index(KEY_COL)['pred'].loc[y_val_true.index].values  - y_val_true.values)**2)))
test_rmse = float(np.sqrt(np.mean((test_unit.set_index(KEY_COL)['pred'].loc[y_test_true.index].values - y_test_true.values)**2)))

print(f'\n[Refit 완료] (reg 단독, y>0 conditional)')
print(f'  best target_transform = {best_target_transform}')
print(f'  best scaling          = {best_scaling}')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')


## 7. 산출물 저장

In [ ]:
refit_result = {
    'oof_pred_die':         oof_pred,
    'val_pred_die':         val_pred,
    'test_pred_die':        test_pred,
    'oof_pi':  None, 'val_pi':  None, 'test_pi':  None,
    'oof_mu':  None, 'val_mu':  None, 'test_mu':  None,
    'oof_pred_unit':        oof_unit,
    'val_pred_unit':        val_unit,
    'test_pred_unit':       test_unit,
    'fold_models':          fold_models,
    'fold_scalers':         fold_scalers,
    'best_params_resolved': {**best_pp,
                             'scaling':          best_scaling,
                             'target_transform': best_target_transform,
                             **best_enet},
    'model_name':           REG_MODEL_NAME,
}

study_meta_for_save = {
    'exp_id':                EXP_ID,
    'exp_memo':              EXP_MEMO,
    'user':                  USER,
    'model_name':            REG_MODEL_NAME,
    'best_target_transform': best_target_transform,
    'best_scaling':          best_scaling,
    'best_pp':               best_pp,
    'y_positive_only':       Y_POSITIVE_ONLY,
    'clip_y_extreme':        CLIP_Y_EXTREME,
    'n_trials':              N_TRIALS,
    'n_folds':               N_FOLDS,
    'n_jobs':                N_JOBS,
    'timeout_sec':           TIMEOUT_SEC,
    'seed_kfold':            SEED,
    'anchor':                ENET_ANCHOR,
    'hpo_best_value':        best_value,
}

hpo.save_artifacts(
    refit_result=refit_result,
    xs_train=xs_train_c, xs_val=xs_val_c, xs_test=xs_test_c,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    postprocess_config=None,   # ★ combine 단계에서 후처리
    study_meta=study_meta_for_save,
)

for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:30s}  {sz:>10,.1f} KB')

try:
    import google.colab
    from google.colab import files
    import shutil
    _zip = shutil.make_archive(os.path.join('/content', f'reg_{REG_MODEL_NAME}_{EXP_ID}_outputs'), 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip} ({os.path.getsize(_zip)/1024:.1f} KB)')
    try:
        files.download(_zip)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크')
        display(FileLink(_zip))
except ImportError:
    pass
